In [ ]:
#r "./../../../public/src/L4-application/BoSSSpad/bin/Release/net6.0/BoSSSpad.dll"
using System;
using System.Threading;
using System.Diagnostics;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Solution;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.BoSSSpad;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Control;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.LoadBalancing;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Utils;
using System.IO;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using MomentFittingVariants = BoSSS.Foundation.XDG.XQuadFactoryHelper.MomentFittingVariants;
Init();

In [ ]:
//defaults
string dbname = System.Environment.GetEnvironmentVariable("DATABASE_NAME");
string buildname = System.Environment.GetEnvironmentVariable("JOB_NAME");
buildname = String.IsNullOrEmpty(buildname)? "RotTorus3D" : buildname; //Popcorn2D  VanishingSphere2D
dbname = String.IsNullOrEmpty(dbname)? "condStudy" : dbname; //Collison´2
string table_name = String.Concat(buildname, "_", dbname);

In [ ]:
using BoSSS.Solution.Gnuplot;


In [ ]:
//ExecutionQueues
var myBatch = GetDefaultQueue();
string WFlowName = table_name;
BoSSS.Application.BoSSSpad.BoSSSshell.WorkflowMgm.Init(WFlowName,myBatch);
BoSSS.Application.BoSSSpad.BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();
var myDB = BoSSS.Application.BoSSSpad.BoSSSshell.WorkflowMgm.DefaultDatabase; 
myDB

In [ ]:
myDB.Sessions.Where(s => s.SuccessfulTermination == true)

In [ ]:
Func<List<double>, double> myOperation = Enumerable.Max; //How to choose the single element from the condition number list
BoSSSshell.WorkflowMgm.condReader.Update(operation: myOperation);


In [ ]:
var Tab = BoSSSshell.WorkflowMgm.SessionTable;

In [ ]:
var Tab2 = Tab.ExtractColumns( "AgglomerationThreshold", "DGdegree:Velocity*","Grid:hMin","Grid:NoOfCells" ,"MassMtxCondNo", "TotCondNo-Vars0.1.2");
Tab2.SaveToFile(table_name + $"_condSummary.json");
Tab2.Print()

In [ ]:
using BoSSS.Solution.Gnuplot;
using System.Linq;

// arrange line color and types
var allColors = Enum.GetValues(typeof(LineColors)).Cast<LineColors>().ToArray();
var allPointTypes = Enum.GetValues(typeof(PointTypes)).Cast<PointTypes>().ToArray();
PointTypes[] myPointTypes = new PointTypes[]{ PointTypes.Diamond, PointTypes.Box,  PointTypes.LowerTriangle,PointTypes.OpenLowerTriangle };

// arrange mesh size and column names
double[] meshes = new double[]{256, 1024, 4096, 16384};

In [ ]:
string varName = "MassMtxCondNo";

Plot2Ddata[,] PlotTable = new Plot2Ddata[2, 2];
int ind = 0;
for (int iCol = 0; iCol < 2; iCol++) {
    for (int iRow = 0; iRow < 2; iRow++) {     
        // Filter data
        var Tab2 = Tab.ExtractColumns( "AgglomerationThreshold", "DGdegree:Velocity*","Grid:hMin","Grid:NoOfCells" ,varName);
        
        // Find the correct mesh resolution for each subfigure
        Tab2 = Tab2.ExtractRows((iRow,RowEntries)=> 
                Convert.ToDouble(RowEntries["Grid:NoOfCells"]) == meshes[ind]
            );

        // Sort in ascending order of AgglomerationThreshold
        System.Data.DataView view = Tab2.DefaultView;
        view.Sort = "DGdegree:Velocity* ASC"; 
        Tab2 = view.ToTable();

        var plot = Tab2.ToPlot("AgglomerationThreshold", varName,  "DGdegree:Velocity*");

        for (int i = 0; i < plot.dataGroups.Length; i++){
            var dataGroup = plot.dataGroups[i];
            dataGroup.Format.LineColor = allColors[i % allColors.Length];
            dataGroup.Format.PointType = myPointTypes[i % myPointTypes.Length];
        }      

        // Plot details
        plot.LogY = true;
        plot.Title = "Grid:NoOfCells=" + meshes[ind]; 
        plot.TitleFont = 20;
        plot.Xlabel = "AggThreshold";  
        plot.Ylabel = varName;  
        plot.SavePgfplotsFile($"{varName}_{meshes[ind]}.tex");
        PlotTable[iCol, iRow] = plot;

        ind++;
    }
}


In [ ]:
PlotTable.PlotNow()

In [ ]:
PlotTable.SaveToGIF(varName + ".png");

In [ ]:

string varName = "TotCondNo-Vars0.1.2.3";

Plot2Ddata[,] PlotTable = new Plot2Ddata[2, 2];
int ind = 0;
for (int iCol = 0; iCol < 2; iCol++) {
    for (int iRow = 0; iRow < 2; iRow++) {     
        // Filter data
        var Tab2 = Tab.ExtractColumns( "AgglomerationThreshold", "DGdegree:Velocity*","Grid:hMin","Grid:NoOfCells" ,varName);
        
        // Find the correct mesh resolution for each subfigure
        Tab2 = Tab2.ExtractRows((iRow,RowEntries)=> 
                Convert.ToDouble(RowEntries["Grid:NoOfCells"]) == meshes[ind]
            );

        // Sort in ascending order of AgglomerationThreshold
        System.Data.DataView view = Tab2.DefaultView;
        view.Sort = "DGdegree:Velocity* ASC"; 
        Tab2 = view.ToTable();
        
        var plot = Tab2.ToPlot("AgglomerationThreshold", varName,  "DGdegree:Velocity*");

        for (int i = 0; i < plot.dataGroups.Length; i++){
            var dataGroup = plot.dataGroups[i];
            dataGroup.Format.LineColor = allColors[i % allColors.Length];
            dataGroup.Format.PointType = myPointTypes[i % myPointTypes.Length];
        }      

        // Plot details
        plot.LogY = true;
        plot.Title = "Grid:NoOfCells=" + meshes[ind]; 
        plot.TitleFont = 20;
        plot.Xlabel = "AggThreshold";  
        plot.Ylabel = varName;  
        plot.SavePgfplotsFile($"{varName}_{meshes[ind]}.tex");
        PlotTable[iCol, iRow] = plot;

        ind++;
    }
}


In [ ]:
PlotTable.PlotNow()

In [ ]:
PlotTable.SaveToGIF(varName + ".png");